In [5]:
!pip install -q google-genai

import json
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.6-flash"

# Enter the key when prompted; it won't be echoed in the cell output.
API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

SYSTEM_TEXT = """
You classify short patient follow-up notes for an outpatient clinic.
Use only the note as evidence. If the reason is unclear, choose "other".
Return the requested structured fields.
"""

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

def build_craft_prompt(note_text):
    return f"""
Context: CareLoop Clinics reviews notes after patients miss appointments.
Role: You are a careful analyst classifying the likely reason for one missed visit.
Action: Classify the note, summarize it in no more than 25 words, and suggest a practical follow-up action. If the reason is unclear, choose "other" rather than guessing.
Tone: Neutral, concise, and respectful.
Format: Return the four fields required by the JSON schema. The root_cause must be exactly one of:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.

Patient note:
{note_text}
"""

def analyse_note(note_text):
    prompt = build_craft_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)

# If your CSV is in the repo's data/ folder, add "data/" to this URL.
notes_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/followup_notes.csv"
)
notes = pd.read_csv(notes_url)

note_n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
result = analyse_note(note_n001)
print(result)

Paste your Gemini API key: ··········
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed her visit after receiving no reminder message and forgetting. She requested to rebook for next week.', 'recommended_action': 'Verify patient contact preferences for reminders and reschedule the appointment for next week.'}


In [6]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google.genai import types

ALLOWED_CAUSES = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
}

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def get_note_reply(note_text):
    """Return both the raw JSON text and its parsed dictionary."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_craft_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )
    raw_text = response.text
    parsed_data = json.loads(raw_text)
    return raw_text, parsed_data

# Use the repo's root URL. If the CSV is in data/, add "data/" before its name.
notes_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/followup_notes.csv"
)
notes = pd.read_csv(notes_url)

results = []
api_calls = 0

for _, row in notes.head(10).iterrows():
    note_text = row["note_text"]

    # Required blank-note check: test NaN before converting to string.
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    # Wait between actual API calls, not before the first call.
    if api_calls > 0:
        time.sleep(13)

    api_calls += 1
    raw_response = ""
    data = {}

    try:
        raw_response, data = get_note_reply(str(note_text).strip())

        # Keep any fields the model returned, even if validation fails.
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_response
        }

        insight = FollowupInsight(**data)
        if insight.root_cause in ALLOWED_CAUSES:
            result["validation_status"] = "valid"

    except (ValidationError, ValueError, TypeError, KeyError) as error:
        # Preserve the raw reply and any fields it contained.
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_response
        }
        print(f"{row['note_id']} validation/JSON error: {error}")

    except Exception as error:
        # Keep the loop running if an API call fails.
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "",
            "confidence": "",
            "summary": "",
            "recommended_action": "",
            "validation_status": "invalid",
            "raw_response": str(error)
        }
        print(f"{row['note_id']} API error: {error}")

    results.append(result)

notes_analysed = pd.DataFrame(results, columns=[
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
])

notes_analysed.to_csv("notes_analysed.csv", index=False)

print("API calls made in this cell:", api_calls)
print("Rows saved:", len(notes_analysed))
print("Saved notes_analysed.csv")
print(notes_analysed[
    ["note_id", "root_cause", "confidence", "validation_status"]
].to_string(index=False))

API calls made in this cell: 9
Rows saved: 10
Saved notes_analysed.csv
note_id            root_cause  confidence validation_status
   N001 forgot_or_no_reminder        0.98             valid
   N002 distance_or_transport        0.95             valid
   N003           cost_or_fee        0.95             valid
   N004                 other        0.00     skipped_blank
   N005  illness_or_emergency        0.95             valid
   N006      work_or_schedule        0.95             valid
   N007 forgot_or_no_reminder        0.95             valid
   N008 distance_or_transport        0.95             valid
   N009                 other        0.90             valid
   N010      work_or_schedule        0.95             valid


In [7]:
from pydantic import ValidationError
import pandas as pd
import time
from google.genai import types

# (a) Test two deliberately invalid replies.
bad_reply_confidence = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "The reason is unclear.",
    "recommended_action": "Ask the patient for more details."
}

bad_reply_missing_field = {
    "root_cause": "other",
    "confidence": 0.5,
    "summary": "The reason is unclear."
    # recommended_action is deliberately missing
}

for label, bad_reply in [
    ("Confidence above 1", bad_reply_confidence),
    ("Missing recommended_action", bad_reply_missing_field)
]:
    try:
        FollowupInsight(**bad_reply)
        print(label, "unexpectedly passed validation")
    except ValidationError as error:
        print(f"\n{label} validation error:")
        print(error)

# For the prompt comparison, keep system instructions and generation
# settings identical. Only the user prompt changes.
comparison_system = "Return JSON that follows the provided schema."

def generate_comparison_reply(prompt):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=comparison_system,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )
    return response.text, json.loads(response.text)

comparison_results = []
api_calls_43 = 0

for note_id in ["N007", "N009"]:
    note_text = notes.loc[
        notes["note_id"] == note_id, "note_text"
    ].iloc[0]

    prompts = {
        "V1": f"Classify this note:\n\n{note_text}",
        "V2": build_craft_prompt(note_text)
    }

    for prompt_name, prompt in prompts.items():
        if api_calls_43 > 0:
            time.sleep(13)

        api_calls_43 += 1
        try:
            raw_text, data = generate_comparison_reply(prompt)
            cause = data.get("root_cause", "")
            comparison_results.append({
                "note_id": note_id,
                "prompt": prompt_name,
                "root_cause": cause,
                "allowed_value": cause in ALLOWED_CAUSES
            })
        except Exception as error:
            comparison_results.append({
                "note_id": note_id,
                "prompt": prompt_name,
                "root_cause": f"ERROR: {error}",
                "allowed_value": False
            })

comparison_table = pd.DataFrame(comparison_results)
print("\nPrompt comparison:")
print(comparison_table.to_string(index=False))
print("\nAPI calls in Part 4.3:", api_calls_43)


Confidence above 1 validation error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Missing recommended_action validation error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...The reason is unclear.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

Prompt comparison:
note_id prompt                 root_cause  allowed_value
   N007     V1   SMS notification failure          False
   N007     V2      forgot_or_no_reminder           True
   N009     V1 Patient phone switched off          False
   N009     V2                      other           True

API calls in Part 4.3: 4


In [8]:
import pandas as pd

clean_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/appointments_clean.csv"
)
clean_appointments = pd.read_csv(clean_url)
notes_analysed = pd.read_csv("notes_analysed.csv")

merged = notes_analysed.merge(
    clean_appointments[
        ["appointment_id", "reminder_sent", "status", "no_show"]
    ],
    on="appointment_id",
    how="left"
)

print("Merged rows:", len(merged))
print("\nNotes by root cause and reminder status:")
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))

print("\nReminder status counts:")
print(merged["reminder_sent"].value_counts(dropna=False))

Merged rows: 10

Notes by root cause and reminder status:
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

Reminder status counts:
reminder_sent
No     5
Yes    5
Name: count, dtype: int64
